# Socle .NET metadata-driven : décoration, sérialisation et prédicat universel

Ce notebook est la **première existence pédagogique** du socle transverse `MyIA.AI.Shared`
(EPIC #7265, tranches A1 + A2). Le socle compile, passe 48 tests — et aucun notebook ne le
référençait. Il démontre le pattern *metadata-driven* en trois moments, du plus mécanique au
plus substantiel :

1. **Décoration → introspection** (A1). On décore un type, on le découvre par réflexion, sans
   aucune inscription explicite.
2. **Sérialisation pilotée par la décoration** (A2). Le même graphe, deux formats, et la
   *décoration seule* qui décide de ce qui sort — avec aller-retour (round-trip).
3. **Le prédicat universel Flee** (la substance). Une règle métier écrite **en chaîne de
   caractères**, compilée à l'exécution, évaluée comme prédicat sur les entités découvertes.
   C'est le moteur d'expressions déclaré en dépendance (`Flee 2.0.0`) et **jamais exercé**
   jusqu'ici : la démonstration du pattern *low-code*, où la règle vient de la donnée, pas du
   code, et change sans recompiler.

> **Prérequis** : le socle doit être buildé une fois (`dotnet build MyIA.AI.Shared/MyIA.AI.Shared.csproj`).
> Le notebook référence l'assembly produite, jamais le code du socle n'est copié-collé.

In [1]:
#r "nuget: Flee, 2.0.0"
#r "nuget: Newtonsoft.Json, 13.0.3"
#r "../../../MyIA.AI.Shared/bin/Debug/net9.0/MyIA.AI.Shared.dll"
using MyIA.AI.ComponentModel.Attributes;
using MyIA.AI.ComponentModel.Entities;
using MyIA.AI.ComponentModel.Providers;
using MyIA.AI.ComponentModel.Serialization;
using Flee.PublicTypes;
using System.Linq;
using System.Collections.Generic;
"Socle MyIA.AI.Shared + Flee 2.0.0 + Newtonsoft.Json 13.0.3 référencés."

The below script needs to be able to find the current output cell; this is an easy method to get it.

Installed Packages Flee, 2.0.0 Newtonsoft.Json, 13.0.3

Socle MyIA.AI.Shared + Flee 2.0.0 + Newtonsoft.Json 13.0.3 référencés.

## 1. Décoration → introspection (A1)

Le contrat tient en une phrase : **décorer un type suffit**. Pas de registre, pas
d'inscription, pas de factory à maintenir. La décoration `[MainCategory("...")]`
suffixe le type d'une catégorie de groupement ; `[AttributeContainer]` marque un type
comme racine d'une hiérarchie d'entités enfants (`IChildEntity`). Le
`ReflectedProviderContainer` parcourt l'assembly et expose ce qui a été décoré.

On définit trois entités d'un domaine e-commerce : une commande (feuille plate,
`ISimpleEntity`) et un arbre catalogue (`IChildEntity` sur deux niveaux).

In [2]:
#nullable enable
[MainCategory("Ventes")]
public sealed class Commande : ISimpleEntity
{
    public string Reference { get; set; } = "";
    public decimal Montant { get; set; }
    public string Pays { get; set; } = "";
    public string Name => Reference;
}

[MainCategory("Catalogue")]
[AttributeContainer]
public sealed class Rayon : IChildEntity
{
    public string Nom { get; set; } = "";
    private readonly List<IChildEntity> _enfants = new();
    public IChildEntity? Parent { get; set; }
    public IReadOnlyList<IChildEntity> Enfants => _enfants;
    IReadOnlyList<IChildEntity> IChildEntity.Children => _enfants;
    public void Ajouter(IChildEntity enfant) { enfant.Parent = this; _enfants.Add(enfant); }
}

[MainCategory("Catalogue")]
public sealed class Categorie : IChildEntity
{
    public string Nom { get; set; } = "";
    private readonly List<IChildEntity> _enfants = new();
    public IChildEntity? Parent { get; set; }
    public IReadOnlyList<IChildEntity> Enfants => _enfants;
    IReadOnlyList<IChildEntity> IChildEntity.Children => _enfants;
    public void Ajouter(IChildEntity enfant) { enfant.Parent = this; _enfants.Add(enfant); }
}
"Entités définies : Commande (ISimpleEntity), Rayon (IChildEntity + AttributeContainer), Categorie (IChildEntity)."

Entités définies : Commande (ISimpleEntity), Rayon (IChildEntity + AttributeContainer), Categorie (IChildEntity).

In [3]:
// Aucune inscription explicite : la décoration suffit.
var provider = ReflectedProviderContainer.FromAssembly<Commande>();

display("Catégories découvertes : " + string.Join(", ", provider.Categories));
display("Entités simples (ISimpleEntity) : " + string.Join(", ", provider.SimpleEntities.Select(t => t.Name)));
display("Conteneurs hiérarchiques ([AttributeContainer]) : " + string.Join(", ", provider.Containers.Select(t => t.Name)));
display("Entités hiérarchiques (IChildEntity) : " + string.Join(", ", provider.ChildEntities.Select(t => t.Name)));
display("Types dans la catégorie [Catalogue] : " + string.Join(", ", provider["Catalogue"].Select(t => t.Name)));

Catégories découvertes : Ventes, Catalogue

Entités simples (ISimpleEntity) : Commande

Conteneurs hiérarchiques ([AttributeContainer]) : Rayon

Entités hiérarchiques (IChildEntity) : Rayon, Categorie

Types dans la catégorie [Catalogue] : Rayon, Categorie

**Lecture.** Aucune ligne n'a enregistré `Commande`, `Rayon` ou `Categorie` auprès d'un
service. La décoration seule les a rendus discoverables, groupés par catégorie, et
classés par rôle (feuille, conteneur, nœud hiérarchique). C'est l'ancre A1 : la
réflexion .NET standard, orchestrée par deux attributs, produit un registre vivant
sans maintenance.

### Exercice 1 — Découvrez votre propre entité

**Objectif.** Ajoutez une entité `Livraison` représentant une expédition, décorez-la
avec `[MainCategory("Logistique")]`, implémentez `ISimpleEntity`, puis reconstruisez
le provider et confirmez qu'elle apparaît dans la catégorie `"Logistique"`.

**Indice.** Suivez le squelette de `Commande` ci-dessus. Le `Name` peut renvoyer un
identifiant de suivi. Recréez `ReflectedProviderContainer.FromAssembly<Livraison>()`
et inspectez `provider["Logistique"]`.

In [4]:
// Exercice 1 : à compléter.
// TODO etudiant : définir [MainCategory("Logistique")] public sealed class Livraison : ISimpleEntity { ... }
// TODO etudiant : reconstruire le provider et afficher provider["Logistique"].
return null;

<null>

## 2. Sérialisation pilotée par la décoration (A2)

Même graphe, format JSON — et c'est encore la **décoration qui décide** de ce qui sort.
Le `MetadataJsonSerializer` round-trippe un arbre `IChildEntity` : le type concret des
enfants est préservé (via `$type`), le back-reference `Parent` — qui formerait un cycle
— est droppé à la sérialisation puis **reconstruit** au chargement. Un graphe sérialisé
puis rechargé reste introspectable exactement comme l'original.

In [5]:
// Un catalogue sur deux niveaux : Rayon -> Categorie -> Categorie.
var multimedia = new Rayon { Nom = "Multimédia" };
var video = new Categorie { Nom = "Vidéo" };
var audio = new Categorie { Nom = "Audio" };
multimedia.Ajouter(video);
multimedia.Ajouter(audio);
video.Ajouter(new Categorie { Nom = "Caméscopes" });

var json = MetadataJsonSerializer.Serialize(multimedia);
display(json);

{
  "Nom": "Multimédia",
  "Enfants": {
    "$type": "System.Collections.Generic.List`1[[MyIA.AI.ComponentModel.Entities.IChildEntity, MyIA.AI.Shared]], System.Private.CoreLib",
    "$values": [
      {
        "$type": "Submission#3+Categorie, ℛ*4d1c3986-52f5-475e-a58b-d7f066f3d6fd#1-3",
        "Nom": "Vidéo",
        "Enfants": {
          "$type": "System.Collections.Generic.List`1[[MyIA.AI.ComponentModel.Entities.IChildEntity, MyIA.AI.Shared]], System.Private.CoreLib",
          "$values": [
            {
              "$type": "Submission#3+Categorie, ℛ*4d1c3986-52f5-475e-a58b-d7f066f3d6fd#1-3",
              "Nom": "Caméscopes",
              "Enfants": {
                "$type": "System.Collections.Generic.List`1[[MyIA.AI.ComponentModel.Entities.IChildEntity, MyIA.AI.Shared]], System.Private.CoreLib",
                "$values": []
              }
            }
          ]
        }
      },
      {
        "$type": "Submission#3+Categorie, ℛ*4d1c3986-52f5-475e-a58b-d7f066f3d6fd

In [6]:
// Round-trip : on recharge le JSON et on vérifie l'intégrité du graphe.
var recharge = MetadataJsonSerializer.Deserialize<Rayon>(json);
var video = (Categorie)recharge.Enfants[0];
var camscopes = (Categorie)video.Enfants[0];

display("Type concret du 1er enfant reconstruit : " + video.GetType().Name);
display("Nom du 1er enfant : " + video.Nom);
display("Parent du 1er enfant re-linké vers la racine ? " + (video.Parent == recharge));
display("Profondeur conservée (Caméscopes sous Vidéo) : " + camscopes.Nom);

Type concret du 1er enfant reconstruit : Categorie

Nom du 1er enfant : Vidéo

Parent du 1er enfant re-linké vers la racine ? True

Profondeur conservée (Caméscopes sous Vidéo) : Caméscopes

**Lecture.** Le `Parent` du premier enfant pointe bien vers `recharge` (la racine
rechargée), pas vers l'arbre original : le cycle a été cassé puis rebâti. Le type
concret (`Categorie`, pas `IChildEntity`) est restauré grâce au discriminateur `$type`.
La sérialisation hérite donc du modèle A1 — un graphe rechargé est redécouvrable par
le même `ReflectedProviderContainer`.

### Exercice 2 — Round-trip d'un graphe plus profond

**Objectif.** Construisez un `Rayon` contenant au moins trois niveaux de `Categorie`,
sérialisez-le, désérialisez-le, puis écrivez une vérification qui descend jusqu'à la
feuille la plus profonde et confirme que chaque nœud a son `Parent` correctement
re-linké.

**Indice.** Une fonction récursive `ProfondeurMax(IChildEntity n)` peut mesurer la
profondeur ; vérifiez `enfant.Parent == parentAttendu` à chaque niveau.

In [7]:
// Exercice 2 : à compléter.
// TODO etudiant : construire un Rayon à 3+ niveaux, le sérialiser, le désérialiser.
// TODO etudiant : vérifier récursivement les back-references Parent.
return null;

<null>

## 3. Le prédicat universel Flee — la substance *low-code*

Les deux premiers moments reposent sur de la réflexion .NET standard. Le troisième est
la raison d'être du pattern *metadata-driven* : **une règle métier écrite en chaîne de
caractères**, compilée à l'exécution par le moteur [Flee](https://github.com/arnonax/Flee),
et évaluée comme prédicat sur les entités découvertes en (1).

Ce que Flee apporte qu'un `if` codé en dur n'apporte pas : la règle vient de la
**donnée** (fichier de config, CSV, saisie d'un utilisateur non-développeur), pas du
code. On la change **sans recompiler** — c'est tout l'enjeu du *low-code*. On compile
l'expression **une fois**, puis on l'évalue contre N entités : c'est un filtre dont la
définition est externe au programme.

In [8]:
// Un jeu de commandes de test, tel que pourrait le fournir une source externe.
var commandes = new List<Commande>
{
    new() { Reference = "C-1001", Montant = 1500m, Pays = "FR" },
    new() { Reference = "C-1002", Montant =  200m, Pays = "FR" },
    new() { Reference = "C-1003", Montant = 3000m, Pays = "BE" },
    new() { Reference = "C-1004", Montant = 1800m, Pays = "FR" },
    new() { Reference = "C-1005", Montant =  900m, Pays = "ES" },
};

// La règle est une CHAÎNE. Elle pourrait venir d'un fichier, d'une DB, d'une UI.
// Grammaire Flee (VB-like) : And/Or/Not, = pour l'égalité, <> pour la différence — pas &&/==.
string regle = "Montant > 1000 And Pays = \"FR\"";

var ctx = new ExpressionContext();
ctx.Variables["Montant"] = 0m;   // type établi AVANT compilation (Flee résout les identifiants à la compilation)
ctx.Variables["Pays"] = "";
var predicat = ctx.CompileDynamic(regle);   // compilé UNE fois

// On évalue le même prédicat contre chaque commande.
var selectionnees = commandes.Where(c =>
{
    ctx.Variables["Montant"] = c.Montant;
    ctx.Variables["Pays"] = c.Pays;
    return (bool)predicat.Evaluate();
}).ToList();

display("Règle compilée : " + regle);
display("Commandes sélectionnées : "
        + string.Join(", ", selectionnees.Select(c => $"{c.Reference} ({c.Montant} {c.Pays})")));

Règle compilée : Montant > 1000 And Pays = "FR"

Commandes sélectionnées : C-1001 (1500 FR), C-1004 (1800 FR)

In [9]:
// Le point clé : changer la règle NE demande AUCUNE recompilation du programme.
// On remplace la chaîne, on recompile l'expression, le reste du pipeline est intact.
string nouvelleRegle = "Montant >= 500 And Montant <= 2000";
var ctx2 = new ExpressionContext();
ctx2.Variables["Montant"] = 0m;
var predicat2 = ctx2.CompileDynamic(nouvelleRegle);

var plageMontant = commandes.Where(c =>
{
    ctx2.Variables["Montant"] = c.Montant;
    return (bool)predicat2.Evaluate();
}).Select(c => $"{c.Reference} ({c.Montant})").ToList();

display("Nouvelle règle (sans toucher au code métier) : " + nouvelleRegle);
display("Sélection : " + string.Join(", ", plageMontant));

Nouvelle règle (sans toucher au code métier) : Montant >= 500 And Montant <= 2000

Sélection : C-1001 (1500), C-1004 (1800), C-1005 (900)

**Lecture.** La première règle (`Montant > 1000 And Pays = "FR"`) sélectionne les
commandes françaises de plus de 1000 ; la seconde (`Montant >= 500 And Montant <= 2000`)
ignore le pays et borne le montant. Aucune des deux n'existe dans le code source au
moment où le programme a été compilé : ce sont des **chaînes**, interprétées à
l'exécution. C'est le *prédicat universel* du patrimoine Aricie (#7265, pépite B4) : le
liant qui transforme une bibliothèque d'introspection en moteur de règles piloté par
la donnée.

Dans une application réelle, ces chaînes vivraient dans un fichier de configuration ou
une base de règles éditée par un fonctionnel — le déploiement d'une nouvelle règle
métier devient une édition de texte, pas une release logicielle.

### Exercice 3 — Votre propre règle compilée

**Objectif.** Écrivez une règle Flee qui sélectionne les commandes dont le montant est
strictement supérieur à la moyenne du jeu de données. Compilez-la, évaluez-la contre
les `commandes`, et affichez les références sélectionnées.

**Indices.** (1) Calculez d'abord `var moyenne = commandes.Average(c => c.Montant);`.
(2) Injectez-la comme variable Flee : `ctx.Variables["Moyenne"] = moyenne;`.
(3) La règle devient la chaîne `"Montant > Moyenne"`. Flee supporte les opérateurs de
comparaison, `&&`, `||`, et les appels à certains membres statiques importés.

In [10]:
// Exercice 3 : à compléter.
// TODO etudiant : calculer la moyenne des montants, l'injecter comme variable Flee,
// TODO etudiant : compiler "Montant > Moyenne", évaluer, afficher les sélectionnées.
return null;

<null>

## Conclusion

Trois moments, une même idée : **la décoration porte la sémantique, le socle fait le
reste**. (1) Décorer un type suffit à le rendre discoverable. (2) La sérialisation
round-trip le graphe en préservant types concrets et hiérarchie. (3) Flee compile une
règle-métier-chaîne en un prédicat réutilisable, ouvrant le pattern *low-code* où la
règle vit dans la donnée, pas dans le code.

Le socle `MyIA.AI.Shared` (EPIC #7265) est désormais visible pédagogiquement. La
prochaine tranche (A2+ XML, `DynamicSurrogate` pour constructeurs non défauts) étendra
le round-trip ; les modules de domaine (Trading : Converter E1, Backtester E2)
s'y brancheront en consommateurs.